# Qwen3-4B — GRPO on ViNumQA, continuing from the SFT checkpoint (Modal, A100-80GB)

Modal / A100-80GB successor to `vsf-grpo-kaggle.ipynb`, which was cramped onto a single T4.
Same core recipe — load the SFT LoRA adapter, continue training it with a program-level PA/EA
reward — but with the T4 workarounds removed and the training loop upgraded to what the extra
VRAM makes possible:

* **In-training eval on the validation set** (`eval_strategy="steps"`) — the T4 version had to
  disable it (`eval_strategy="no"`) because generating eval completions on top of the train
  batch OOMed. On 80GB it fits, so a rising/falling **eval reward** is now visible during training.
* **Early stopping** on `eval_reward` (`load_best_model_at_end`, `EarlyStoppingCallback`) — mirrors
  the SFT notebooks' early-stopping, but keyed on reward rather than eval_loss. GRPO's loss is a
  policy-gradient surrogate that doesn't track answer quality, whereas eval reward here *is* the
  PA/EA scorer averaged over the valid set, so it's a near-proxy for validation PA.
* **One epoch** (`num_train_epochs=1`) instead of a fixed `max_steps`.
* **Bigger batch / more frequent eval** to use the 80GB card (`gradient_accumulation_steps=4`
  restored from the T4's forced-down 2).

Removed as unnecessary on Modal/A100: Part A's standalone sanity-check (it double-loaded the
model into VRAM), the T4-specific vllm/triton pinning branch, and `UNSLOTH_VLLM_STANDBY`
(a VRAM-saving trick not needed with headroom to spare, and the source of the `libnvrtc.so.13`
error on Modal's image). Install is handled by the notebook's custom Modal image, so there is no
`pip install` cell here at all.


### Load the SFT checkpoint (not the base model)

In [1]:
!mkdir /root/adapter /root/dataset
!huggingface-cli download ntphuc149/Qwen3-4B-STaNR --local-dir /root/adapter

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 11 files:   0%|                                | 0/11 [00:00<?, ?it/s]Still waiting to acquire lock on /root/adapter/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /root/adapter/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

special_tokens_map.json: 100%|████████████████| 614/614 [00:00<00:00, 1.90MB/s]
Download complete. Moving file to /root/adapter/special_tokens_map.json

merges.txt: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]


chat_template.jinja: 4.67kB [00:00, 4.06MB/s]
.gitattributes: 1.57kB [00:00, 1.81MB/s]Download complete. Moving file to /root/adapter/chat_template.jinja

Download complete. Moving file to /root/adapter/.gitattributes


README.md: 5.24kB [00:00, 10.2MB/s]
Download complete. Moving file to /root/adapter/README.md


added_tokens.json: 100%|██████████████████████| 707/707 [00:00<00:00, 1.66MB/s]
Dow

In [2]:
from unsloth import FastLanguageModel
import torch, os, json
from safetensors.torch import load_file
from peft import set_peft_model_state_dict

MAX_SEQ_LENGTH = 4567  # matches the SFT notebooks -- see their MAX_SEQ_LENGTH measurement cells
LORA_RANK = 32

ADAPTER_DIR = "/root/adapter"
# ^ the "SFT (w ENG trace; PA match only)" checkpoint -- GRPO continues training from here, it
# does not start from the base model like Unsloth's own GSM8K template.
assert os.path.isdir(ADAPTER_DIR), f"ADAPTER_DIR does not exist: {ADAPTER_DIR}"

# fast_inference=True (vLLM) does NOT support pointing model_name straight at a LoRA adapter
# directory the way plain transformers loading does -- vLLM always loads the *base* model first
# (this is why the load log below will correctly say "unsloth/qwen3-4b", not the adapter path).
# So: read the base model name + LoRA hyperparameters out of the adapter's own config (the
# source of truth for what it was actually trained with, rather than hardcoding a guess),
# load that base model, recreate the identical LoRA structure with get_peft_model(), then load
# the trained adapter weights into that structure directly.
with open(os.path.join(ADAPTER_DIR, "adapter_config.json")) as f:
    adapter_cfg = json.load(f)
BASE_MODEL_NAME = adapter_cfg["base_model_name_or_path"]
LORA_RANK = adapter_cfg["r"]
LORA_ALPHA = adapter_cfg["lora_alpha"]
TARGET_MODULES = adapter_cfg["target_modules"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,       # False for LoRA 16bit, matching Unsloth's GRPO template
    fast_inference = True,      # vLLM fast batched generation, needed for GRPO's group sampling
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = 0.7,  # leave headroom for the reference-policy KL computation
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Load the trained adapter's weights into the freshly created LoRA structure. Without this
# step, the checkpoint loaded above has the *shape* of the SFT adapter but randomly initialized
# weights -- Part A's sanity-check would silently be probing an un-fine-tuned model.
adapter_state_dict = load_file(os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))
set_peft_model_state_dict(model, adapter_state_dict)
print(f"Loaded SFT adapter weights from {ADAPTER_DIR} (base: {BASE_MODEL_NAME}, r={LORA_RANK}).")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING 08-12 07:13:16 [config.py:71] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 08-12 07:13:48 [vllm_utils.py:739] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.8.15: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.23.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen3-4b with actual GPU utilization = 69.6%
Unsloth: Your GPU has CUDA compute capabilit

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.37s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.08s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.13s/it]


INFO 08-12 07:14:50 [default_loader.py:397] Loading weights took 2.28 seconds
INFO 08-12 07:14:51 [punica_selector.py:20] Using PunicaWrapperGPU.


...

Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'norm', 'post_layernorm', 'norm2', 'q_norm', 'layer_norm2', 'post_per_layer_input_norm', 'layer_norm1', 'post_feedforward_layernorm', 'norm1']


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 20.94it/s]
Some weights of Qwen3ForCausalLM were not initialized from the model checkpoint at unsloth/qwen3-4b and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'post_attention_layernorm', 'cross_attn_post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'norm', 'post_layernorm', 'norm2', 'q_norm', 'cross_attn_input_layernorm', 'layer_norm2', 'post_per_layer_input_norm', 'layer_norm1', 'post_feedforward_layernorm', 'norm1']


Unsloth 2026.8.15 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Loaded SFT adapter weights from /root/adapter (base: unsloth/qwen3-4b-unsloth-bnb-4bit, r=32).


### Shared prompt format + scorer (same as every other notebook in this repo)

In [3]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""


In [4]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.
See notebooks/evaluate/scorer.py for the full docstring and design rationale.
Inlined here so this notebook has no external file dependency on Kaggle/Modal.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


def str_to_num(text: str) -> Union[float, str]:
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


def program_tokenization(original_program: str) -> List[str]:
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching \')\' found) in program: \'{original_program}\'"
            )

        program.append(m.group(1) + "(")
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: \'{text[pos:]}\' (from: \'{original_program}\')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text


def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps


def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


def equal_program(program1: List[str], program2: List[str]) -> bool:
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


def _coerce_answer(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


### Reward function (tiered PA > EA-only > invalid, with a capped conciseness penalty)

In [5]:
R_PA = 1.0
R_EA_ONLY = 0.3
R_INVALID = 0.0
CONCISE_PENALTY_PER_STEP = 0.05
CONCISE_PENALTY_CAP_FRACTION = 0.15  # penalty never exceeds this fraction of the base reward


def compute_reward(generated_program: str, gold_program: str, gold_answer, table) -> float:
    pa, ea = score_one(generated_program, gold_program, gold_answer, table)

    if pa == 1.0:
        base = R_PA
    elif ea == 1.0:
        base = R_EA_ONLY
    else:
        base = R_INVALID  # covers both "doesn\'t parse" and "parses but wrong" -- GRPO\'s
                           # within-group relative advantage doesn\'t need a finer split here

    if base > 0:
        try:
            n_steps_gen = len(program_tokenization(extract_program(generated_program))) // 4
        except ValueError:
            n_steps_gen = None
        n_steps_gold = len(program_tokenization(gold_program)) // 4
        if n_steps_gen is not None and n_steps_gen > n_steps_gold:
            penalty = CONCISE_PENALTY_PER_STEP * (n_steps_gen - n_steps_gold)
            base -= min(penalty, base * CONCISE_PENALTY_CAP_FRACTION)

    return base


### Data prep

`test_df` is loaded here (used by the final test-set eval); `formatting_*` helpers are shared with the dataset builder below.

In [6]:
import pandas as pd
from tabulate import tabulate
import random

random.seed(3407)

def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

# test_df is needed by the final test-set eval cell at the end of the notebook, so it is loaded
# here even though Part A's sanity-check (which also used it) has been removed on Modal.
test_df = pd.read_json("/root/dataset/test.json")
test_df["pre_text_f"] = test_df.apply(formatting_pre_text, axis=1)
test_df["table_f"] = test_df.apply(formatting_table, axis=1)
test_df["post_text_f"] = test_df.apply(formatting_post_text, axis=1)
test_df["table_raw"] = test_df["table"]
test_df["question"] = test_df["qa"].apply(lambda x: x["question"])
test_df["program"] = test_df["qa"].apply(lambda x: x["program"])
test_df["answer"] = test_df["qa"].apply(lambda x: x["exe_ans"])


In [7]:
from datasets import Dataset

TRAIN_JSON_PATH = "/root/dataset/train.json"
# ^ adjust: this should match whichever training set produced the checkpoint loaded in Part A,
# so GRPO continues refining behavior on data the model has already seen the "shape" of.

VALID_JSON_PATH = "/root/dataset/valid.json"
# ^ datasets/ViNumQA/distill-from-gemma-teacher/pa-match-only/eng/valid.json -- the teacher-solved
# subset of valid.json, matching whichever variant TRAIN_JSON_PATH points to (NOT
# datasets/ViNumQA/origin/valid.json -- the SFT checkpoint this notebook continues from was
# itself only ever trained/evaluated on the teacher-solved subset). Not wired into GRPOTrainer
# as eval_dataset (that OOMs on a single T4 -- see GRPOConfig's eval_strategy comment below);
# used instead by the standalone post-training validation-set eval cell near the end.

def build_prompt(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text_f"], table=row["table_f"],
        post_text=row["post_text_f"], question=row["question"],
    )
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]

def build_grpo_dataset(json_path):
    raw = pd.read_json(json_path)
    raw["pre_text_f"] = raw.apply(formatting_pre_text, axis=1)
    raw["table_f"] = raw.apply(formatting_table, axis=1)
    raw["post_text_f"] = raw.apply(formatting_post_text, axis=1)
    raw["table_raw"] = raw["table"]
    raw["question"] = raw["qa"].apply(lambda x: x["question"])
    raw["program"] = raw["qa"].apply(lambda x: x["program"])
    raw["answer"] = raw["qa"].apply(lambda x: x["exe_ans"])
    raw["prompt"] = raw.apply(build_prompt, axis=1)
    return raw

grpo_train_raw = build_grpo_dataset(TRAIN_JSON_PATH)
grpo_valid_raw = build_grpo_dataset(VALID_JSON_PATH)  # kept as a DataFrame for the post-training eval cell

grpo_dataset = Dataset.from_pandas(
    grpo_train_raw[["prompt", "program", "answer", "table_raw"]].rename(
        columns={"program": "gold_program", "answer": "gold_answer"}
    ).reset_index(drop=True)
)
print(f"GRPO training set: {len(grpo_dataset)} samples")
print(f"GRPO validation set (for post-training eval only): {len(grpo_valid_raw)} samples")
grpo_dataset[0]

GRPO training set: 2123 samples
GRPO validation set (for post-training eval only): 450 samples


{'prompt': [{'role': 'system',
   'content': 'You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.\n\n### LIST OF 10 VALID OPERATORS:\n\n1. add(a, b) -> a + b\n2. subtract(a, b) -> a - b\n3. multiply(a, b) -> a * b\n4. divide(a, b) -> a / b\n5. exp(a, b) -> a^b\n6. greater(a, b) -> 1.0 if a > b, else 0.0\n7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`\n8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`\n9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`\n10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`\n\n### RULES:\n- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.\n- table_* operators take exactly two argu

In [8]:
# In-training eval needs the validation split as an HF Dataset (the Kaggle notebook only kept it
# as a DataFrame, since eval was disabled there). Same four columns as grpo_dataset.
grpo_valid_dataset = Dataset.from_pandas(
    grpo_valid_raw[["prompt", "program", "answer", "table_raw"]].rename(
        columns={"program": "gold_program", "answer": "gold_answer"}
    ).reset_index(drop=True)
)
print(f"GRPO validation set (HF Dataset for in-training eval): {len(grpo_valid_dataset)} samples")


GRPO validation set (HF Dataset for in-training eval): 450 samples


In [9]:
def program_reward_func(prompts, completions, gold_program, gold_answer, table_raw, **kwargs):
    """GRPOTrainer reward function: one score per completion.

    completions[i] corresponds to gold_program[i]/gold_answer[i]/table_raw[i] (TRL passes
    dataset columns through as same-named kwargs, broadcast across the group). Splits off the
    </think> reasoning trace the same way Part A did, so only the program half is scored.
    """
    scores = []
    for completion, gold_prog, gold_ans, table in zip(completions, gold_program, gold_answer, table_raw):
        raw_text = completion[0]["content"]
        close_idx = raw_text.find("</think>")
        program_text = raw_text[close_idx + len("</think>"):].strip() if close_idx != -1 else raw_text.strip()
        scores.append(compute_reward(program_text, gold_prog, gold_ans, table))
    return scores


<a name="Train"></a>
### Train the model — 1 epoch, in-training eval + early stopping

In [10]:
from vllm import SamplingParams
from trl import GRPOConfig, GRPOTrainer
from transformers import EarlyStoppingCallback

max_prompt_length = 2048   # measure actual prompt lengths first if unsure -- see the SFT
                            # notebooks' MAX_SEQ_LENGTH-measurement cells for the pattern
max_completion_length = MAX_SEQ_LENGTH - max_prompt_length

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=3407,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

# Full-valid eval is expensive here (~37 min/eval: all ~450 valid prompts, each sampled
# num_generations=8 times). One epoch is ~530 optimizer steps, so evaluating every 100 steps
# gives ~5 evals -- enough for early stopping to act on while keeping total eval time bounded.
effective_batch = 8 * 4  # per_device_train_batch_size * gradient_accumulation_steps
import math
steps_per_epoch = math.ceil(len(grpo_dataset) / effective_batch)
eval_every = 100         # fixed: ~5 evals across the single ~530-step epoch
print(f"~{steps_per_epoch} optimizer steps / epoch; evaluating every {eval_every} steps "
      f"(~{max(1, steps_per_epoch // eval_every)} evals this epoch)")

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,

    per_device_train_batch_size=8,     # 80GB has room; T4 was stuck at 1
    gradient_accumulation_steps=4,     # restored from the T4's forced-down 2 (effective batch 32)
    num_generations=8,                 # G -- group size

    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,

    num_train_epochs=1,                # single epoch, per the plan (no fixed max_steps)
    output_dir="/root/qwen3-4b-vinumqa-grpo-adapter-ckpts",
    report_to="none",

    # In-training eval on the fixed validation set -- now affordable on 80GB. Its metric is the
    # same program reward the trainer optimizes, averaged over valid, so it doubles as a PA proxy.
    per_device_eval_batch_size=8,      # must be a multiple of num_generations
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,             # must match eval cadence for load_best_model_at_end
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_reward",   # TRL logs the aggregate eval reward under this key
    greater_is_better=True,                # higher reward = better (unlike eval_loss)
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[program_reward_func],
    args=training_args,
    train_dataset=grpo_dataset,
    eval_dataset=grpo_valid_dataset,       # HF Dataset version, built in the data-prep cell
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    # ^ stop if eval_reward hasn't improved for 2 consecutive evals; load_best_model_at_end then
    #   restores the best-reward checkpoint before the post-training eval cells below run.
)


~67 optimizer steps / epoch; evaluating every 100 steps (~1 evals this epoch)


[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Run the cell below to start training. Watch **eval reward** (not train reward, and not loss) — that is the number early stopping and best-checkpoint selection key on.

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,123 | Num Epochs = 1 | Total steps = 530
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


WARNING 08-12 07:18:54 [input_processor.py:157] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
WARNING 08-12 07:19:00 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _rms_layernorm_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 08-12 07:19:01 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _rope_embedding_QK. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 08-12 07:19:01 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _fg_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 08-12 07:21:57 [jit_monitor.py:103] Triton kernel JIT com

Step,Training Loss,Validation Loss


### Saving

In [ ]:
ADAPTER_OUT_DIR = "/root/qwen3-4b-vinumqa-grpo-adapter"
model.save_pretrained(ADAPTER_OUT_DIR)   # best-reward checkpoint (load_best_model_at_end restored it)
tokenizer.save_pretrained(ADAPTER_OUT_DIR)
print(f"Saved best GRPO LoRA adapter to {ADAPTER_OUT_DIR}")


### Validation-set PA/EA (single pass, post-training)

In [ ]:
FastLanguageModel.for_inference(model)

greedy_sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    max_tokens=1024,
)

valid_prompts = grpo_valid_raw["prompt"].apply(
    lambda messages: tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
    )
).tolist()

valid_outputs = model.fast_generate(valid_prompts, sampling_params=greedy_sampling_params)

valid_pa_scores, valid_ea_scores = [], []
for row, output in zip(grpo_valid_raw.itertuples(), valid_outputs):
    raw_text = output.outputs[0].text
    close_idx = raw_text.find("</think>")
    program_text = raw_text[close_idx + len("</think>"):].strip() if close_idx != -1 else raw_text.strip()
    pa, ea = score_one(program_text, row.program, row.answer, row.table_raw)
    valid_pa_scores.append(pa)
    valid_ea_scores.append(ea)

valid_program_accuracy = sum(valid_pa_scores) / len(valid_pa_scores)
valid_execution_accuracy = sum(valid_ea_scores) / len(valid_ea_scores)
print(f"Validation set: {len(grpo_valid_raw)} samples")
print(f"Program Accuracy (PA): {valid_program_accuracy:.4f}")
print(f"Execution Accuracy (EA): {valid_execution_accuracy:.4f}")


### Final PA/EA on the held-out test set

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = []
for _, row in test_df.iterrows():
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text_f"], table=row["table_f"],
        post_text=row["post_text_f"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    test_prompts.append(text)

test_outputs = model.fast_generate(test_prompts, sampling_params=greedy_sampling_params)

pa_scores, ea_scores = [], []
for row, output in zip(test_df.itertuples(), test_outputs):
    raw_text = output.outputs[0].text
    close_idx = raw_text.find("</think>")
    program_text = raw_text[close_idx + len("</think>"):].strip() if close_idx != -1 else raw_text.strip()
    pa, ea = score_one(program_text, row.program, row.answer, row.table_raw)
    pa_scores.append(pa)
    ea_scores.append(ea)

program_accuracy = sum(pa_scores) / len(pa_scores)
execution_accuracy = sum(ea_scores) / len(ea_scores)
print(f"Test set: {len(test_df)} samples")
print(f"Program Accuracy (PA): {program_accuracy:.4f}")
print(f"Execution Accuracy (EA): {execution_accuracy:.4f}")